# Phase IV: Model Evaluation Strategy

In this phase, we move beyond subjective testing and use quantitative metrics to evaluate our model's performance. We will specifically use **ROUGE (Recall-Oriented Understudy for Gisting Evaluation)**, which measures the overlap between a machine-generated summary and a human-written reference.

### What we will do:
1. **Load the Test Data**: We will use a subset of the `cnn_dailymail` test split.
2. **Generate Predictions**: Run our BART model on these articles.
3. **Compute ROUGE Scores**: Compare predictions against the human "highlights".
4. **Analyze Results**: Understand how well the model captures key information.

## 1. Setup and Environment

We need to load our model, tokenizer, and the `evaluate` library.

In [ ]:
import yaml
import os
import pandas as pd
from datasets import load_dataset
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
import evaluate
import torch
from tqdm.auto import tqdm

# Load configuration
config_path = os.path.join("..", "config.yaml")
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

model_name = config['model']['name']
dataset_name = config['data']['dataset_name']
dataset_config = config['data']['dataset_config']

# Load metrics
rouge = evaluate.load("rouge")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## 2. Loading a Subset of the Test Data

The CNN/DailyMail dataset has over 11,000 test samples. To keep this evaluation fast, we will sample 50 random articles.

In [ ]:
print("Loading dataset...")
test_data = load_dataset(dataset_name, dataset_config, split="test")

# Selecting a small subset for quick evaluation
sample_size = 50
sample_data = test_data.shuffle(seed=42).select(range(sample_size))

print(f"Selected {len(sample_data)} samples for evaluation.")

## 3. Running Inference and Scoring

We will loop through the subset, generate summaries, and store them for scoring.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

predictions = []
references = []

print("Generating summaries...")
for example in tqdm(sample_data):
    article = example['article']
    reference = example['highlights']
    
    # Generate summary
    inputs = tokenizer(article, max_length=1024, truncation=True, return_tensors="pt").to(device)
    summary_ids = model.generate(inputs['input_ids'], num_beams=4, max_length=128, early_stopping=True)
    prediction = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    
    predictions.append(prediction)
    references.append(reference)

print("Inference complete.")

## 4. Calculating ROUGE Scores

ROUGE-1, ROUGE-2, and ROUGE-L are the standard metrics for summarization.

In [ ]:
results = rouge.compute(predictions=predictions, references=references)
print("\n--- Evaluation Results ---")
for key, value in results.items():
    print(f"{key}: {value:.4f}")

### Interpretation of Results:
- **ROUGE-1**: Overlap of individual words (unigrams).
- **ROUGE-2**: Overlap of two-word phrases (bigrams).
- **ROUGE-L**: Longest common subsequence between the generated and reference text.

Typically, for BART on CNN/DM, you should see ROUGE-1 scores around 0.40 - 0.44.